In [1]:
import pandas as pd
import numpy as np
import json

In [2]:
df = pd.read_csv("../data/aircraft engine/PM_train.csv")

In [3]:
df.shape

(20631, 26)

In [4]:
df.id.nunique()

100

In [5]:
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
df.cycle.value_counts()

cycle
8      100
9      100
10     100
11     100
12     100
      ... 
359      1
360      1
361      1
353      1
362      1
Name: count, Length: 362, dtype: int64

In [7]:
gdf = df[['id', 'cycle']].groupby('id').count()

In [8]:
gdf.sort_values(by = 'cycle', ascending=True)

,cycle
id,
39,128
91,135
57,137
70,137
24,147
...,...
83,293
67,313
96,336


In [9]:
df.id.describe()

count    20631.000000
mean        51.506568
std         29.227633
min          1.000000
25%         26.000000
50%         52.000000
75%         77.000000
max        100.000000
Name: id, dtype: float64

In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20631 entries, 0 to 20630
Data columns (total 26 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        20631 non-null  int64  
 1   cycle     20631 non-null  int64  
 2   setting1  20631 non-null  float64
 3   setting2  20631 non-null  float64
 4   setting3  20631 non-null  float64
 5   s1        20631 non-null  float64
 6   s2        20631 non-null  float64
 7   s3        20631 non-null  float64
 8   s4        20631 non-null  float64
 9   s5        20631 non-null  float64
 10  s6        20631 non-null  float64
 11  s7        20631 non-null  float64
 12  s8        20631 non-null  float64
 13  s9        20631 non-null  float64
 14  s10       20631 non-null  float64
 15  s11       20631 non-null  float64
 16  s12       20631 non-null  float64
 17  s13       20631 non-null  float64
 18  s14       20631 non-null  float64
 19  s15       20631 non-null  float64
 20  s16       20631 non-null  fl

In [11]:
df['max'] = df.groupby('id')['cycle'].transform('max')
df["RUL"] = df['max'] - df['cycle']

In [12]:
df.to_parquet("../data/aircraft engine/PM_train.parquet")

In [13]:
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import GradientBoostingRegressor
import xgboost as xgb
from sklearn.metrics import accuracy_score, r2_score, mean_squared_error
from scipy.stats import loguniform

In [14]:
x = df.drop(['id', 'cycle', 'RUL', 'max'], axis=1)

In [15]:
y = df['RUL']

In [16]:
try:
    with open('note.json', 'r') as file:
        info = json.loads(file.read())
except FileNotFoundError:
    info = {}
info

{'gb_model': {'r2_score': 0.8666493068005445,
  'best_params': {'subsample': 1.0,
   'n_estimators': 2000,
   'min_samples_split': 5,
   'min_samples_leaf': 2,
   'max_depth': 5,
   'loss': 'squared_error',
   'learning_rate': 0.1,
   'criterion': 'friedman_mse',
   'alpha': 0.5}},
 'gb_model_2': {'r2_score': 0.8653454963351543,
  'best_params': {'alpha': 0.7,
   'criterion': 'friedman_mse',
   'learning_rate': 0.07536840425014499,
   'loss': 'squared_error',
   'max_depth': 9,
   'min_samples_leaf': 3,
   'min_samples_split': 7,
   'n_estimators': 2000,
   'subsample': 0.8}}}

In [17]:
def GBR(x,y):
    x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.21, random_state=42)
    gbr_params = {
    'loss':['squared_error', 'absolute_error', 'huber', 'quantile'],
    'learning_rate':loguniform(1e-4, 100),
    'n_estimators':[1000, 2000, 3000],
    'subsample':[0.8, 0.6],
    'criterion':['friedman_mse', 'squared_error'],
    'min_samples_split':[3, 4, 5, 7, 9],
    'min_samples_leaf':[2, 3, 4, 5],
    'max_depth':[None, 3, 5, 7, 9, 11],
    'alpha':[0.9, 0.7],
    }
    gbr_model = GradientBoostingRegressor(n_iter_no_change=15, random_state=42)
    _gbr_model = RandomizedSearchCV(estimator=gbr_model, param_distributions=gbr_params, cv=3, n_iter=50, verbose=2)
    _gbr_model.fit(x_train, y_train)
    return _gbr_model

In [19]:
gb_model = GBR(x,y)

Fitting 3 folds for each of 50 candidates, totalling 150 fits
[CV] END alpha=0.9, criterion=friedman_mse, learning_rate=19.80718475282676, loss=huber, max_depth=5, min_samples_leaf=5, min_samples_split=5, n_estimators=2000, subsample=0.6; total time=   0.5s
[CV] END alpha=0.9, criterion=friedman_mse, learning_rate=19.80718475282676, loss=huber, max_depth=5, min_samples_leaf=5, min_samples_split=5, n_estimators=2000, subsample=0.6; total time=   0.4s
[CV] END alpha=0.9, criterion=friedman_mse, learning_rate=19.80718475282676, loss=huber, max_depth=5, min_samples_leaf=5, min_samples_split=5, n_estimators=2000, subsample=0.6; total time=   0.4s
[CV] END alpha=0.9, criterion=squared_error, learning_rate=98.46239670994154, loss=squared_error, max_depth=3, min_samples_leaf=5, min_samples_split=9, n_estimators=2000, subsample=0.6; total time=   0.2s
[CV] END alpha=0.9, criterion=squared_error, learning_rate=98.46239670994154, loss=squared_error, max_depth=3, min_samples_leaf=5, min_samples_sp

KeyboardInterrupt: 

In [ ]:
gb_model.best_params_

In [25]:
x_train,x_test,y_train,y_test = train_test_split(x,y, test_size=0.21, random_state=42)

In [ ]:
y_pred_gb = gb_model.predict(x_test)

In [ ]:
r2_score(y_test, y_pred_gb)

In [ ]:
# mean_squared_error(y_test, y_pred)

In [ ]:
info.update({'gb_model_3':{'r2_score':r2_score(y_test, y_pred_gb), 'best_params':gb_model.best_params_}})

In [ ]:


# learning_rate = loguniform(1e-4, 100)

# # Generate 10 random samples from the distribution
# samples = learning_rate.rvs(10)
# print(samples)


In [ ]:
# import numpy as np

# # Create an array of x values (sampled on a log scale for better visualization)
# x = np.logspace(np.log10(1e-4), np.log10(100), num=10)


# print(x)          # x values


In [ ]:
# import json
# with open('note.json', 'w') as file:
#     file.write(json.dumps(info))

In [22]:
def xgboost(x,y):
    xgb_params = {
        
    'n_estimators': [1000, 2000, 3000],               # Set high for early stopping
    'learning_rate': loguniform(1e-4, 100),  # Step size shrinkage
    'max_depth': [None, 3, 5, 7, 10],              # Tree complexity
    'subsample': [0.8, 1.0],             # Rows per tree
    'colsample_bytree': [0.8, 1.0],      # Columns per tree
    'grow_policy': ['depthwise', 'lossguide'],
    # Regularization
    'reg_alpha': [0, 0.1, 1, 2],
    'reg_lambda': [0.1, 1, 10, 20],
    'gamma': [0, 0.1, 1],
    # unbalanced
    # 'min_child_weight': [1, 5, 10],
    # 'eval_metric': ['logloss', 'aucpr']  # Handle class imbalance
        
}
    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=42)
    x_temp, x_val, y_temp, y_val = train_test_split(x_train, y_train, test_size=0.25, random_state=42)
    xgb_model = xgb.XGBRegressor(early_stopping_rounds=10, eval_metric='mae')
    grid_xgb_model = RandomizedSearchCV(estimator=xgb_model, param_distributions=xgb_params, cv=3, n_iter=30, scoring='r2', verbose=1)
    grid_xgb_model.fit(x_temp, y_temp, eval_set=[(x_val, y_val)])
    return grid_xgb_model

In [23]:
xgb_model = xgboost(x,y)

Fitting 3 folds for each of 30 candidates, totalling 90 fits
[0]	validation_0-mae:558.47887
[1]	validation_0-mae:6418.33015
[2]	validation_0-mae:74294.79951
[3]	validation_0-mae:861244.74275
[4]	validation_0-mae:10006656.21334
[5]	validation_0-mae:116317381.16153
[6]	validation_0-mae:1352861410.86439
[7]	validation_0-mae:15727805862.98273
[8]	validation_0-mae:183197116553.75482
[9]	validation_0-mae:2130268845980.94019
[0]	validation_0-mae:555.71191
[1]	validation_0-mae:6424.01842
[2]	validation_0-mae:74498.89661
[3]	validation_0-mae:868187.09406
[4]	validation_0-mae:10150964.65480
[5]	validation_0-mae:118588671.88691
[6]	validation_0-mae:1385289381.45199
[7]	validation_0-mae:16170152812.57138
[8]	validation_0-mae:189288064299.43973
[9]	validation_0-mae:2213877676932.81543
[10]	validation_0-mae:25921840244483.25000
[0]	validation_0-mae:550.95050
[1]	validation_0-mae:6337.14450
[2]	validation_0-mae:73553.62198
[3]	validation_0-mae:854917.18627
[4]	validation_0-mae:9941297.46024
[5]	valid

In [27]:
y_pred_xgb = xgb_model.predict(x_test)

In [28]:
r2_score(y_test, y_pred_xgb)

0.633815586566925

In [29]:
info.update({'xgb_model_2':{'r2_score':r2_score(y_test, y_pred_xgb), 'best_params':xgb_model.best_params_}})

In [31]:
import json
with open('note.json', 'w') as file:
    file.write(json.dumps(info))

In [30]:
info

{'gb_model': {'r2_score': 0.8666493068005445,
  'best_params': {'subsample': 1.0,
   'n_estimators': 2000,
   'min_samples_split': 5,
   'min_samples_leaf': 2,
   'max_depth': 5,
   'loss': 'squared_error',
   'learning_rate': 0.1,
   'criterion': 'friedman_mse',
   'alpha': 0.5}},
 'gb_model_2': {'r2_score': 0.8653454963351543,
  'best_params': {'alpha': 0.7,
   'criterion': 'friedman_mse',
   'learning_rate': 0.07536840425014499,
   'loss': 'squared_error',
   'max_depth': 9,
   'min_samples_leaf': 3,
   'min_samples_split': 7,
   'n_estimators': 2000,
   'subsample': 0.8}},
 'xgb_model_2': {'r2_score': 0.633815586566925,
  'best_params': {'colsample_bytree': 1.0,
   'gamma': 0.1,
   'grow_policy': 'depthwise',
   'learning_rate': 0.010724528148632458,
   'max_depth': None,
   'n_estimators': 2000,
   'reg_alpha': 0.1,
   'reg_lambda': 0.1,
   'subsample': 0.8}}}